# Loading and Inspecting Data

**Topic:** Exploratory Data Analysis

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import Dropdown, Output, HBox, VBox, HTML

from IPython.display import display, clear_output
from scipy import stats

titanic = sns.load_dataset("titanic")
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this session you will be able to:

- **Use** the core pandas inspection methods to get an immediate overview of any new dataset
- **Distinguish** between what `info()` tells you and what `describe()` tells you
- **Identify** missing values and unexpected data types on first contact with the data

> **Tip:** Work through every method in the dropdown in order. By the time you reach `describe()`, notice how much more you know about the data than you did after just running `head()`.

---
## How we got here

In `01_what_is_eda.ipynb` we introduced the Titanic dataset and framed the central question: what factors influenced survival? Now it is time to actually open the data and take a first look. Before you can analyze anything, you need to know what you are working with.

---
## Why this matters for data science

The first five minutes with a new dataset set the direction for everything that follows. Experienced data scientists run a fixed set of inspection commands the moment they load any data, because the answers tell them where to focus. A single call to `info()` can reveal that a column you planned to use is 40% missing, saving hours of wasted modeling work.

---
## Try it yourself

In [ ]:
out = Output()

METHODS = {
    "head()": {
        "description": "Returns the first 5 rows. Use this to get a quick sense of what the data looks like.",
        "fn": lambda df: df.head(),
    },
    "tail()": {
        "description": "Returns the last 5 rows. Useful for checking if the data ends cleanly or if something odd happens at the bottom.",
        "fn": lambda df: df.tail(),
    },
    "info()": {
        "description": "Shows column names, non-null counts, and data types. This is the fastest way to spot missing data and type issues.",
        "fn": lambda df: df.info(),
    },
    "describe()": {
        "description": "Computes summary statistics (count, mean, std, min, max, quartiles) for all numeric columns.",
        "fn": lambda df: df.describe(),
    },
    "shape": {
        "description": "Returns a tuple (rows, columns). The very first thing to check on any new dataset.",
        "fn": lambda df: df.shape,
    },
    "dtypes": {
        "description": "Shows the data type of every column. Mismatched types (e.g. numbers stored as strings) are a common source of bugs.",
        "fn": lambda df: df.dtypes,
    },
    "value_counts('survived')": {
        "description": "Counts how many times each unique value appears in the survived column. Essential for understanding class balance.",
        "fn": lambda df: df["survived"].value_counts(),
    },
}

method_dropdown = Dropdown(
    options=list(METHODS.keys()),
    value="head()",
    description="Method:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"),
)

def render(method_name):
    entry = METHODS[method_name]
    desc_html = HTML(
        f'<div style="font-family: Inter, Arial, sans-serif; font-size: 14px; '
        f'color: #495057; padding: 10px 14px; background: #EEF2FF; '
        f'border-left: 4px solid #4C6EF5; border-radius: 4px; margin-bottom: 8px;">'
        f'{entry["description"]}</div>'
    )
    with out:
        clear_output(wait=True)
        display(desc_html)
        result = entry["fn"](titanic)
        if result is not None:
            display(result)

def on_change(change):
    render(method_dropdown.value)

method_dropdown.observe(on_change, names="value")

display(VBox([method_dropdown, out]))
render(method_dropdown.value)

---
## What's happening?

Loading data is one line of code. Understanding data takes longer. The pandas inspection methods form a toolkit for that understanding:

| Method | What it answers |
|--------|----------------|
| `head()` / `tail()` | What do individual rows look like? Are there any obvious formatting issues? |
| `shape` | How many rows and columns do I have? |
| `dtypes` | Are columns stored as the types I expect? |
| `info()` | Which columns have missing values? How large is the dataset in memory? |
| `describe()` | What is the spread and center of each numeric column? |
| `value_counts()` | How are categorical values distributed? Are any values dominant? |

Notice that `info()` and `describe()` tell completely different stories. `info()` is about structure and completeness. `describe()` is about the statistical distribution of values. You need both.

---
## Real-world example: Titanic Dataset

The chart below shows how many non-null values exist in each column. Columns reaching the full 891 are complete. Columns falling short have missing data that will need to be addressed.

Notice:
- **`age`** has 714 non-null values, meaning 177 passengers have no recorded age
- **`cabin`** has only 204 non-null values, making it roughly 77% missing
- **`embarked`** is nearly complete with just 2 missing values

> **Discussion question:** Which columns have missing values? Which ones surprise you? Why might cabin information be missing for so many passengers?

In [ ]:
non_null_counts = titanic.notnull().sum().sort_values()

fig = go.Figure(
    data=[go.Bar(
        x=non_null_counts.values,
        y=non_null_counts.index.tolist(),
        orientation="h",
        marker_color=[
            PALETTE["secondary"] if v < 891 else PALETTE["accent"]
            for v in non_null_counts.values
        ],
        text=[f"{v} / 891" for v in non_null_counts.values],
        textposition="outside",
    )],
    layout=base_layout(
        title="Non-Null Count per Column (Titanic, n=891)",
        xaxis_title="Number of Non-Null Values",
        yaxis_title="",
    ),
)
fig.update_layout(
    height=480,
    xaxis=dict(range=[0, 980]),
    showlegend=False,
)
fig.show()

### Common first-inspection surprises across real datasets

| Dataset | Surprising finding on first look |
|---------|----------------------------------|
| Titanic passenger data | Cabin is 77% missing, but survived is complete |
| Medical records | Age recorded as strings like "35 years" not integers |
| E-commerce orders | Negative quantities (returns mixed with purchases) |
| Survey responses | Likert scale stored as text ("Agree") not numbers |
| Web log data | Timestamps stored as Unix epoch integers, not datetime |

---
## Key takeaway

> **Always run `info()` and `describe()` before touching a single line of analysis code, because what you find in those two outputs determines every decision that follows.**

---
*Next up: 03_data_types_and_casting.ipynb — examining the types of each column and learning when and how to convert them*